# LLM2-Lab — ferramenta opcional para A100

Este notebook é um roteiro de infraestrutura; A100/BF16 não foram empiricamente validados
pela campanha 0015, que usou T4 FP32. Treino e avaliações dependem dos switches explícitos.
O fluxo de pesquisa atual e seus limites estão em `docs/PROJECT_STATUS.md`.

Para preparar o código: `python scripts/package_colab.py --output dist/llm2-source.zip`.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import zipfile

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import torch
assert torch.cuda.is_available(), 'Selecione um runtime GPU no Colab.'
properties = torch.cuda.get_device_properties(0)
assert 'A100' in properties.name, f'Este protocolo pede A100; encontrada: {properties.name}'
assert properties.total_memory >= 39 * 1024**3, 'É necessária A100 com pelo menos 40 GB nominais.'
assert torch.cuda.is_bf16_supported(), 'BF16 indisponível.'
print(torch.__version__, torch.version.cuda, properties.name, properties.total_memory / 1024**3, 'GiB')


In [ ]:
from google.colab import files
ROOT = Path('/content/LLM2-Lab')
if not (ROOT / 'pyproject.toml').exists():
    uploaded = files.upload()
    archives = [name for name in uploaded if name.endswith('.zip')]
    assert len(archives) == 1, 'Envie um único ZIP criado por scripts/package_colab.py.'
    ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for member in archive.namelist():
            target = (ROOT / member).resolve()
            assert target.is_relative_to(ROOT.resolve()), 'Caminho inválido no ZIP.'
        archive.extractall(ROOT)
os.chdir(ROOT)
# Usa o Torch CUDA já presente se ele satisfizer o pyproject; não instala wheel CPU.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-e', '.[dev,research]'])

def run(*args):
    subprocess.check_call([sys.executable, *map(str, args)], cwd=ROOT)

run('-m', 'pytest', '-q', '-m', 'not cuda')


Persistência: habilite `USE_DRIVE` para gravar dados, métricas e checkpoints no Google Drive. Sem Drive, baixe os artefatos antes de encerrar o runtime. Checkpoints são diretórios imutáveis; no Drive/FUSE, interrupções de I/O ainda podem exigir repetir a cópia. Os checksums detectam cópias incompletas.


In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE = Path('/content/drive/MyDrive/LLM2-Lab')
else:
    STORAGE = ROOT
STORAGE.mkdir(parents=True, exist_ok=True)
print('Artefatos em', STORAGE)


## Research choices and resource limits

Credit rates and balances in the budget examples are historical planning inputs, not a current
account balance or pricing quote. Update them before budgeting a session. The watchdog checks
between optimizer steps; it does not terminate the runtime or guarantee a billing ceiling.
Training and evaluation remain explicitly opt-in.


In [ ]:
import yaml
CONFIG_TEMPLATE = 'configs/profiles/a100_7m.yaml'
ARCHITECTURE = 'transformer'  # recursive; fractal requires the experimental branch ZIP
DATASET_PATH = ''             # existing prepared data, or empty for synthetic demo
RUN_NAME = 'research-v2-demo'
TRAIN_SEED = 42
MAX_STEPS = 20
FIRST_STAGE = 10
MAX_SECONDS = 1800.0
SHUTDOWN_RESERVE_SECONDS = 60.0
MAX_INPUT_TOKENS = 100000
MAX_LEARNING_FLOPS = 0       # 0 disables this analytical cap
COMPILE = False
DETERMINISTIC = False
RUN_TRAINING = False
RUN_EVALUATIONS = False
RUN_CUDA_SMOKE = False
EXPORT_RESULTS = False
template = yaml.safe_load((ROOT / CONFIG_TEMPLATE).read_text(encoding='utf-8'))
print('No paid training is started by configuration or inspection.')


In [ ]:
from llm2.data import PreparedData
IS_BYTE = template['model']['vocab_size'] == 257
RAW = STORAGE / 'data/raw/demo.txt'
DATA = Path(DATASET_PATH) if DATASET_PATH else STORAGE / ('data/processed-byte' if IS_BYTE else 'data/processed-bpe')
if not DATASET_PATH:
    if not RAW.exists():
        run('scripts/make_demo_corpus.py', '--output', RAW)
    if not DATA.exists():
        run('-m', 'llm2', 'prepare', '--input', RAW, '--output', DATA,
            '--tokenizer', 'byte' if IS_BYTE else 'bpe', '--vocab-size', '8192', '--seed', '42')
prepared = PreparedData(DATA)
print('Data hash:', prepared.fingerprint)
print({key: value['tokens'] for key, value in prepared.manifest['splits'].items()})


Demonstracao curta, parada voluntaria e retomada. O horizonte do scheduler vem do YAML selecionado e permanece fixo. Use RUN_TRAINING e RUN_EVALUATIONS explicitamente; avaliacoes adicionais e exportacao ficam fora do watchdog do Trainer. COMPILE e DETERMINISTIC continuam opcoes de protocolo.


In [ ]:
from llm2.config import ExperimentConfig
from llm2.research.budget import plan_budget
config = template
config['data']['path'] = str(DATA)
config['train'].update(output_dir=str(STORAGE / 'runs' / RUN_NAME), seed=TRAIN_SEED,
                       max_steps=MAX_STEPS, eval_interval=10, checkpoint_interval=10,
                       log_interval=1, compile=COMPILE, deterministic=DETERMINISTIC)
config.setdefault('research', {}).update(architecture=ARCHITECTURE, max_seconds=MAX_SECONDS,
    shutdown_reserve_seconds=SHUTDOWN_RESERVE_SECONDS, max_tokens=MAX_INPUT_TOKENS,
    max_train_flops=MAX_LEARNING_FLOPS)
checked = ExperimentConfig.from_dict(config)
assert not checked.research.planning_only, 'Planning-only profiles cannot train.'
assert checked.model.vocab_size >= prepared.tokenizer.vocab_size
assert min(len(a) for a in prepared.arrays.values()) > checked.data.seq_len
CONFIG = ROOT / 'configs' / f'colab-{RUN_NAME}.yaml'
serialized = yaml.safe_dump(config)
if CONFIG.exists() and CONFIG.read_text(encoding='utf-8') != serialized:
    raise FileExistsError('Config changed: choose a new RUN_NAME or restore the exact prior configuration.')
if not CONFIG.exists():
    CONFIG.write_text(serialized, encoding='utf-8')
run('-m', 'llm2', 'inspect', '--config', CONFIG, '--meta')
print(plan_budget(session_minutes=MAX_SECONDS / 60))
if RUN_CUDA_SMOKE:
    run('-m', 'pytest', '-q', '-m', 'cuda')


In [ ]:
import json
RUN = Path(config['train']['output_dir'])
LATEST = RUN / 'checkpoints/latest.json'
checkpoint = None
if not RUN_TRAINING:
    print('Training disabled. Set RUN_TRAINING=True only when ready to use GPU credits.')
else:
    if not LATEST.exists():
        run('-m', 'llm2', 'train', '--config', CONFIG, '--until-step', FIRST_STAGE)
    checkpoint = RUN / 'checkpoints' / json.loads(LATEST.read_text())['checkpoint']
    run('-m', 'llm2', 'train', '--config', CONFIG, '--resume', checkpoint)
    checkpoint = RUN / 'checkpoints' / json.loads(LATEST.read_text())['checkpoint']
    print(json.loads((RUN / 'summary.json').read_text()))


In [ ]:
if RUN_EVALUATIONS and checkpoint is not None:
    run('-m', 'llm2', 'evaluate', '--checkpoint', checkpoint, '--data', DATA,
        '--device', 'cuda', '--precision', 'bf16')
    run('-m', 'llm2', 'generate', '--checkpoint', checkpoint, '--data', DATA,
        '--device', 'cuda', '--precision', 'bf16', '--prompt', 'O laboratorio',
        '--max-new-tokens', 40, '--seed', 42)
    EVAL = RUN / 'evaluations' / checkpoint.name
    EVAL.mkdir(parents=True, exist_ok=True)
    if not (EVAL / 'benchmark.json').exists():
        run('-m', 'llm2', 'benchmark', '--checkpoint', checkpoint, '--data', DATA,
            '--device', 'cuda', '--precision', 'bf16', '--max-new-tokens', 16,
            '--output', EVAL / 'benchmark.json')
    if not (EVAL / 'reasoning.json').exists():
        run('-m', 'llm2', 'reasoning', '--checkpoint', checkpoint, '--data', DATA,
            '--device', 'cuda', '--precision', 'bf16', '--examples', 1,
            '--output', EVAL / 'reasoning.json')
    REPORT = RUN / 'reports' / checkpoint.name
    if not REPORT.exists():
        run('-m', 'llm2', 'report', '--runs', RUN, '--output', REPORT)
if EXPORT_RESULTS and checkpoint is not None:
    destination = STORAGE / 'exports' / f'{RUN_NAME}-{checkpoint.name}.zip'
    if not destination.exists():
        run('scripts/export_results.py', '--run', RUN, '--output', destination)
    print('Logs/configs/reports exported; weights remain in the persistent run:', destination)


Métricas: `metrics.jsonl`, `summary.json` e `tensorboard/` no diretório da execução. Para visualizar no Colab, execute a célula opcional abaixo. Tempos de GPU usam eventos CUDA por optimizer-step; não representam soma dos tempos individuais de kernels. Não compare o corpus sintético com benchmarks de linguagem.


In [ ]:
SHOW_TENSORBOARD = False
if SHOW_TENSORBOARD:
    ipython = get_ipython()
    ipython.run_line_magic('load_ext', 'tensorboard')
    ipython.run_line_magic('tensorboard', f'--logdir {RUN / "tensorboard"}')
